# Crypto Swarm — Night Shift Optimizer

Runs the full night_shift pipeline on Colab. Data and results sync via Google Drive.

**Runtime:** ~35-45 min for 4 symbols

### Data Flow
```
Binance API → Colab Session → Google Drive ← Local Machine (Drive sync)
                              ↳ data/ohlcv/     ↳ night_results/YYYY-MM-DD/
```

---
## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/gdrive')

import os

# Project root on Drive — results will appear here via Drive sync
DRIVE_ROOT = '/gdrive/My Drive/crypto-swarm-trader'
DATA_DIR = os.path.join(DRIVE_ROOT, 'data', 'ohlcv')
RESULTS_DIR = os.path.join(DRIVE_ROOT, 'data', 'night_results')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Data dir:  {DATA_DIR}')
print(f'Results dir: {RESULTS_DIR}')

---
## 2. Configuration

In [ ]:
# @title Pipeline Configuration
SYMBOLS = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'BNB/USDT']  # @param
FETCH_FRESH_DATA = True  # @param {type:"boolean"}
NUM_FOLDS = 9  # @param
TEST_FOLD_DAYS = 36  # @param
DARWINIAN_GENERATIONS = 5  # @param
DARWINIAN_POPULATION = 50  # @param

print(f'Symbols: {SYMBOLS}')
print(f'Folds: {NUM_FOLDS}, Test days: {TEST_FOLD_DAYS}')
print(f'Darwinian: {DARWINIAN_GENERATIONS} gen x {DARWINIAN_POPULATION} pop')

---
## 3. Install Dependencies & Fetch Data

In [ ]:
!pip install -q pandas numpy ccxt

import pandas as pd
import numpy as np
import asyncio
from datetime import datetime, timedelta, timezone

if FETCH_FRESH_DATA:
    import ccxt.async_support as ccxt_async

    async def fetch_all():
        exchange = ccxt_async.binance({'enableRateLimit': True})
        since = int((datetime.now(timezone.utc) - timedelta(days=365)).timestamp() * 1000)
        for symbol in SYMBOLS:
            safe = symbol.replace('/', '_')
            all_candles = []
            while len(all_candles) < 9000:
                batch = await exchange.fetch_ohlcv(symbol, '1h', since=since, limit=1000)
                if not batch: break
                all_candles.extend(batch)
                since = batch[-1][0] + 1
                await asyncio.sleep(exchange.rateLimit / 1000)
            if all_candles:
                df = pd.DataFrame(all_candles, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
                df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
                df = df.drop_duplicates(subset='timestamp').set_index('timestamp').sort_index()
                df.to_parquet(os.path.join(DATA_DIR, f'{safe}_1h.parquet'))
                print(f'  {symbol}: {len(df)} candles')
        await exchange.close()

    print('Fetching from Binance...')
    await fetch_all()
    print('Done.')

# Verify
for symbol in SYMBOLS:
    safe = symbol.replace('/', '_')
    path = os.path.join(DATA_DIR, f'{safe}_1h.parquet')
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f'  {symbol}: {len(df)} candles ({df.index[0].date()} to {df.index[-1].date()})')
    else:
        print(f'  {symbol}: MISSING')

---
## 4. Core Pipeline (self-contained, no repo needed)

In [ ]:
from typing import Dict, List
from itertools import product
from collections import Counter
from dataclasses import dataclass, asdict, field
import time, random, json

PRODUCTION_CONFIG = {
    'signal_threshold': 0.40, 'min_alignment': 3, 'take_profit_atr': 6.0,
    'stop_loss_atr': 2.5, 'max_hold_hours': 96, 'time_decay_hours': 48,
    'trailing_stop_atr': 1.0, 'score_flip_delay_hrs': 2,
}
COARSE_GRID = {
    'signal_threshold': [0.30, 0.33, 0.35, 0.38, 0.40, 0.43, 0.45, 0.50],
    'take_profit_atr': [3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 7.0],
    'stop_loss_atr': [1.0, 1.25, 1.5, 2.0, 2.5, 3.0],
    'max_hold_hours': [36, 48, 72, 96, 120],
    'time_decay_hours': [12, 24, 36, 48],
    'trailing_stop_atr': [0.0, 1.0],
    'score_flip_delay_hrs': [0, 2],
}
FINE_GRID = {
    'trailing_stop_atr': [0.0, 0.5, 0.8, 1.0, 1.2, 1.5],
    'score_flip_delay_hrs': [0, 1, 2, 3, 4],
}
SHARPE_CAP = 100.0
OVERFITTING_CONFIG = {'max_is_oos_gap': 0.5, 'min_oos_consistency': 0.50}

@dataclass
class Fold:
    fold_num: int; train_start_idx: int; train_end_idx: int
    test_start_idx: int; test_end_idx: int; train_hours: int; test_hours: int

@dataclass
class CandidateResult:
    symbol: str; params: Dict; oos_sharpe: float; oos_pnl: float; oos_pf: float
    oos_wr: float; oos_max_dd: float; oos_consistency: float; oos_avg_trades_per_fold: float
    oos_mean_hold_hrs: float; oos_exit_reasons: Dict; is_sharpe: float; is_pnl: float
    overfitting_score: float; fragility: float; survivor_score: float
    folds: List[Dict] = field(default_factory=list)
    rejected: bool = False; rejection_reason: str = ''; is_coarse_only: bool = False

def log(msg): print(f'[{datetime.now(timezone.utc).strftime("%H:%M:%S")}] {msg}', flush=True)

def compute_indicators(df):
    close, high, low, volume = df['close'], df['high'], df['low'], df['volume']
    for name, lookback in [('1h', 20), ('4h', 80), ('1d', 200)]:
        sma = close.rolling(lookback).mean()
        df[f'trend_{name}'] = np.where(close > sma, 1, np.where(close < sma, -1, 0))
    df['bullish_count'] = ((df['trend_1h'] == 1).astype(int) + (df['trend_4h'] == 1).astype(int) + (df['trend_1d'] == 1).astype(int))
    df['bearish_count'] = ((df['trend_1h'] == -1).astype(int) + (df['trend_4h'] == -1).astype(int) + (df['trend_1d'] == -1).astype(int))
    delta = close.diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    sma20 = close.rolling(20).mean(); std20 = close.rolling(20).std()
    df['bb_upper'] = sma20 + 2*std20; df['bb_lower'] = sma20 - 2*std20; df['bb_middle'] = sma20
    df['bb_pos'] = (close - df['bb_lower']) / (4 * std20)
    df['momentum_4h'] = close.pct_change().rolling(80).mean()
    df['vol_ratio'] = volume / volume.rolling(20).mean()
    tr = pd.concat([high - low, (high - close.shift()).abs(), (low - close.shift()).abs()], axis=1).max(axis=1)
    df['atr'] = tr.rolling(14).mean()
    df['volatility'] = close.pct_change().rolling(20).std()
    return df

def _compute_score(bull, bear, rsi, mom, bb_pos, vol_ratio, threshold, min_alignment):
    score = 0.0
    if bull >= min_alignment: score += (bull/3.0)*0.4; 
    elif bear >= min_alignment: score -= (bear/3.0)*0.4
    if bull >= min_alignment and vol_ratio > 1.3: score += 0.1
    elif bear >= min_alignment and vol_ratio > 1.3: score -= 0.1
    if np.isnan(rsi): rsi = 50
    if rsi < 30: score += 0.3
    elif rsi < 35 and bull >= min_alignment: score += 0.2
    elif rsi > 70: score -= 0.3
    elif rsi > 65 and bear >= min_alignment: score -= 0.2
    if np.isnan(mom): mom = 0
    if mom > 0.003: score += 0.15
    elif mom < -0.003: score -= 0.15
    if np.isnan(bb_pos): bb_pos = 0.5
    if bb_pos < 0.15: score += 0.15
    elif bb_pos > 0.85: score -= 0.15
    return score

def simulate_trades(df, params):
    threshold = params['signal_threshold']; min_alignment = params.get('min_alignment', 3)
    tp_atr_mult = params['take_profit_atr']; sl_atr_mult = params['stop_loss_atr']
    max_hold = params['max_hold_hours']; decay_hours = params['time_decay_hours']
    trailing_atr = params.get('trailing_stop_atr', 0)
    flip_delay = params.get('score_flip_delay_hrs', 0)
    trips = []; in_position = False; entry_price = entry_idx = 0
    entry_rsi = 50; peak_price = 0
    close = df['close'].values; atr = df['atr'].values; rsi = df['rsi'].values
    bull = df['bullish_count'].values; bear = df['bearish_count'].values
    mom = df['momentum_4h'].values; bb = df['bb_pos'].values; vol = df['vol_ratio'].values
    for i in range(250, len(close)):
        price = close[i]; a = atr[i] if not np.isnan(atr[i]) else 0
        r = rsi[i] if not np.isnan(rsi[i]) else 50
        if in_position:
            hold = i - entry_idx; pnl = (price - entry_price) / entry_price * 100
            if price > peak_price: peak_price = price
            if trailing_atr > 0 and a > 0 and peak_price > entry_price:
                trail = trailing_atr * a / entry_price * 100
                if (peak_price - price) / entry_price * 100 >= trail:
                    trips.append({'pnl_pct': (peak_price-entry_price)/entry_price*100 - trail, 'hold_hrs': hold, 'exit': 'trailing_stop'})
                    in_position = False; continue
            if a > 0 and pnl <= -(sl_atr_mult * a / entry_price * 100):
                trips.append({'pnl_pct': pnl, 'hold_hrs': hold, 'exit': 'stop_loss'}); in_position = False; continue
            if a > 0 and pnl >= (tp_atr_mult * a / entry_price * 100):
                trips.append({'pnl_pct': pnl, 'hold_hrs': hold, 'exit': 'take_profit'}); in_position = False; continue
            if hold >= max_hold:
                trips.append({'pnl_pct': pnl, 'hold_hrs': hold, 'exit': 'max_hold'}); in_position = False; continue
            if pnl < 0 and hold >= decay_hours:
                trips.append({'pnl_pct': pnl, 'hold_hrs': hold, 'exit': 'time_decay'}); in_position = False; continue
            sc = _compute_score(bull[i], bear[i], r, mom[i], bb[i], vol[i], threshold, min_alignment)
            if sc < 0 and (flip_delay == 0 or hold >= flip_delay):
                trips.append({'pnl_pct': pnl, 'hold_hrs': hold, 'exit': 'score_flip'}); in_position = False; continue
            if r > 55 and entry_rsi < 35:
                trips.append({'pnl_pct': pnl, 'hold_hrs': hold, 'exit': 'mr_target'}); in_position = False; continue
        else:
            sc = _compute_score(bull[i], bear[i], r, mom[i], bb[i], vol[i], threshold, min_alignment)
            if sc > threshold:
                in_position = True; entry_price = price; entry_idx = i; entry_rsi = r; peak_price = price
    return trips

def compute_metrics(trips):
    if not trips: return {'round_trips': 0, 'win_rate': 0, 'total_pnl_pct': 0, 'pf': 0, 'sharpe': 0, 'max_dd_pct': 0, 'avg_hold_hrs': 0, 'exit_reasons': {}}
    pnls = [t['pnl_pct'] for t in trips]
    wins = [p for p in pnls if p > 0]; losses = [p for p in pnls if p <= 0]
    cum = np.cumsum(pnls); running_max = np.maximum.accumulate(cum)
    max_dd = abs(min(cum - running_max))
    sharpe = np.mean(pnls) / np.std(pnls) * np.sqrt(24*365) if np.std(pnls) > 0 else 0
    avg_win = np.mean(wins) if wins else 0; avg_loss = abs(np.mean(losses)) if losses else 0
    pf = avg_win / avg_loss if avg_loss > 0 else 999
    exits = {}
    for t in trips: exits[t.get('exit', 'signal')] = exits.get(t.get('exit', 'signal'), 0) + 1
    return {'round_trips': len(trips), 'win_rate': len(wins)/len(pnls), 'total_pnl_pct': sum(pnls),
            'pf': pf, 'sharpe': sharpe, 'max_dd_pct': max_dd,
            'avg_hold_hrs': np.mean([t['hold_hrs'] for t in trips]), 'exit_reasons': exits}

def create_folds(total_bars, num_folds, test_fold_days):
    test_bars = test_fold_days * 24; warmup = 250
    if total_bars <= warmup + test_bars:
        return [Fold(0, 0, warmup, warmup, total_bars, warmup, total_bars - warmup)]
    usable = total_bars - warmup
    actual = min(num_folds, usable // test_bars)
    bars_per = test_bars if actual < (usable // test_bars) else usable // actual
    folds = []; start = warmup
    for i in range(actual):
        end = start + bars_per if i < actual - 1 else total_bars
        folds.append(Fold(i, 0, start, start, end, start, end - start))
        start = end
    return folds

def eval_fold(df, fold, params, skip_is=False):
    if not skip_is:
        td = df.iloc[fold.train_start_idx:fold.train_end_idx]
        tt = simulate_trades(td, params) if len(td) > 250 else []
        tm = compute_metrics(tt); is_s = tm['sharpe']; is_p = tm['total_pnl_pct']
    else: is_s = 0.0; is_p = 0.0
    ed = df.iloc[fold.test_start_idx:fold.test_end_idx]
    et = simulate_trades(ed, params) if len(ed) > 10 else []
    em = compute_metrics(et)
    return {'is_sharpe': is_s, 'is_pnl': is_p, 'oos_sharpe': em['sharpe'], 'oos_pnl': em['total_pnl_pct'],
            'oos_pf': em['pf'], 'oos_wr': em['win_rate'], 'oos_max_dd': em['max_dd_pct'],
            'oos_trades': em['round_trips'], 'oos_avg_hold': em['avg_hold_hrs'],
            'oos_exit_reasons': em.get('exit_reasons', {})}

def eval_candidate(df, folds, params, symbol, of_cfg, compute_frag=False, skip_is=False):
    fr = [eval_fold(df, f, params, skip_is=skip_is) for f in folds]
    raw = [f['oos_sharpe'] for f in fr]
    ws = [max(-SHARPE_CAP, min(SHARPE_CAP, s)) for s in raw]
    oos_pfs = [f['oos_pf'] for f in fr if f['oos_pf'] < 999]
    is_s = [f['is_sharpe'] for f in fr]; is_p = [f['is_pnl'] for f in fr]
    avg_is = float(np.mean(is_s)) if is_s else 0
    avg_oos = float(np.median(ws)) if ws else 0
    avg_pnl = float(np.sum([f['oos_pnl'] for f in fr]))
    avg_pf = float(np.mean(oos_pfs)) if oos_pfs else 0
    avg_wr = float(np.mean([f['oos_wr'] for f in fr]))
    avg_dd = float(np.mean([f['oos_max_dd'] for f in fr]))
    avg_tr = float(np.mean([f['oos_trades'] for f in fr]))
    avg_hld = float(np.mean([f['oos_avg_hold'] for f in fr]))
    cons = sum(1 for s in ws if s > 0) / len(ws) if ws else 0
    ax = Counter()
    for f in fr:
        for r, c in f['oos_exit_reasons'].items(): ax[r] += c
    if avg_is == 0: of_score = 0.0
    elif abs(avg_is) > 0.01: of_score = max(0, (avg_is - avg_oos) / abs(avg_is))
    else: of_score = 0.5 if avg_oos < 0 else 0.0
    is_coarse = len(fr) < 3
    frag = 0.0
    if compute_frag and avg_oos > 0.1:
        for pn, pv in params.items():
            if pn == 'min_alignment' or not isinstance(pv, (int, float)): continue
            for d in [-0.10, 0.10]:
                pert = {**params, pn: round(pv * (1 + d), 4)}
                ps = max(-SHARPE_CAP, min(SHARPE_CAP, eval_fold(df, folds[-1], pert)['oos_sharpe']))
                if abs(avg_oos) > 0.01: frag = max(frag, abs(ps - avg_oos) / abs(avg_oos))
    of_pen = 1.0 - min(of_score, 1.0)
    dd_fac = 1.0 / (1.0 + avg_dd / 100)
    tr_fac = min(avg_tr / max(of_cfg.get('min_trades_per_fold', 10), 1), 1.0)
    f_pen = 1.0 / (1.0 + frag)
    surv = avg_oos * cons * of_pen * dd_fac * tr_fac * f_pen
    rej = False; rej_r = ''
    if of_score > of_cfg.get('max_is_oos_gap', 0.5): rej = True; rej_r = f'overfitting={of_score:.2f}'
    if cons < of_cfg.get('min_oos_consistency', 0.50): rej = True; rej_r = f'consistency={cons:.0%}'
    fd = [{'fold': folds[i].fold_num, 'is_sharpe': f['is_sharpe'], 'oos_sharpe': ws[i],
           'oos_sharpe_raw': raw[i], 'oos_pnl': f['oos_pnl'], 'oos_trades': f['oos_trades']} for i, f in enumerate(fr)]
    return CandidateResult(symbol, dict(params), avg_oos, avg_pnl, avg_pf, avg_wr, avg_dd, cons, avg_tr, avg_hld,
                            dict(ax), avg_is, float(np.sum(is_p)), of_score, frag, surv, fd, rej, rej_r, is_coarse)

def grid_combos(g): return [dict(zip(g.keys(), c)) for c in product(*g.values())]

def coarse_search(df, folds, sym, of_cfg):
    combos = grid_combos(COARSE_GRID); log(f'  Coarse: {len(combos)} for {sym}')
    w = 720
    cf = Fold(0, 0, max(0, len(df)-w-250), len(df)-w, len(df), max(0, len(df)-w-250), w) if len(df) > w else folds[-1]
    res = []
    for i, c in enumerate(combos):
        p = {**c, 'min_alignment': 3}
        res.append(eval_candidate(df, [cf], p, sym, of_cfg, skip_is=True))
        if (i+1) % 10000 == 0: log(f'    {i+1}/{len(combos)} best={max(r.survivor_score for r in res):.3f}')
    log(f'    {sum(1 for r in res if not r.rejected)}/{len(combos)} passed')
    return res

def fine_refine(df, folds, sym, top, of_cfg):
    res = []; seen = set()
    for p in top:
        if p.rejected: continue
        b = dict(p.params); b.pop('trailing_stop_atr', None); b.pop('score_flip_delay_hrs', None)
        for c in grid_combos(FINE_GRID):
            pm = {**b, **c}; k = tuple(sorted(pm.items()))
            if k not in seen: seen.add(k); res.append(eval_candidate(df, folds, pm, sym, of_cfg, compute_frag=True))
    log(f'    Fine: {len(res)} on {len(folds)} folds')
    return res

def darwinian(df, folds, sym, pop, of_cfg, gens=5, ps=50):
    cg = sorted([r for r in pop if not r.rejected], key=lambda r: r.survivor_score, reverse=True)[:ps]
    if not cg: return []
    all_s = list(cg)
    for g in range(gens):
        off = []
        for par in cg:
            for _ in range(3):
                pm = dict(par.params)
                nk = [k for k, v in pm.items() if isinstance(v, (int, float)) and k != 'min_alignment']
                if not nk: continue
                k = random.choice(nk); d = random.uniform(0.05, 0.15) * random.choice([-1, 1])
                o = pm[k]; pm[k] = max(1, int(o*(1+d))) if isinstance(o, int) else max(0.01, round(o*(1+d), 4))
                off.append(eval_candidate(df, folds, pm, sym, of_cfg, compute_frag=True))
        cg = sorted(cg + off, key=lambda r: r.survivor_score, reverse=True)[:ps]
        all_s.extend(cg)
        log(f'    Gen {g+1}/{gens}: {len(off)} offspring best={cg[0].survivor_score:.3f}')
    seen = set(); uniq = []
    for r in sorted(all_s, key=lambda r: r.survivor_score, reverse=True):
        k = tuple(sorted(r.params.items()))
        if k not in seen: seen.add(k); uniq.append(r)
    return uniq[:ps*2]

print('Pipeline loaded OK')

---
## 5. Run Night Shift

In [ ]:
t0 = time.time()
log('=' * 60)
log('NIGHT SHIFT — Autonomous Strategy Optimization')
log(f'Symbols: {SYMBOLS}')
log('=' * 60)

# Load
dfs = {}
for sym in SYMBOLS:
    safe = sym.replace('/', '_')
    p = os.path.join(DATA_DIR, f'{safe}_1h.parquet')
    if not os.path.exists(p): log(f'WARNING: no data for {sym}'); continue
    dfs[sym] = compute_indicators(pd.read_parquet(p))
    log(f'  {sym}: {len(dfs[sym])} candles')

# Folds
folds = create_folds(min(len(d) for d in dfs.values()), NUM_FOLDS, TEST_FOLD_DAYS)
log(f'{len(folds)} folds created')

# Phase 2b: Baseline
all_results = {}
log('Phase 2b: Production Baseline')
for sym in dfs:
    cr = eval_candidate(dfs[sym], folds, PRODUCTION_CONFIG, sym, OVERFITTING_CONFIG, compute_frag=True)
    all_results.setdefault(sym, []).append(cr)
    log(f'  {sym}: Sharpe={cr.oos_sharpe:+.2f} cons={cr.oos_consistency:.0%} surv={cr.survivor_score:.2f}')

# Phase 3: Coarse
log('Phase 3: Coarse Grid Search')
for sym in dfs:
    all_results.setdefault(sym, []).extend(coarse_search(dfs[sym], folds, sym, OVERFITTING_CONFIG))

# Phase 3b: Fine
log('Phase 3b: Fine Refinement')
for sym in list(all_results):
    top = sorted(all_results[sym], key=lambda r: r.survivor_score, reverse=True)[:100]
    all_results[sym].extend(fine_refine(dfs[sym], folds, sym, top, OVERFITTING_CONFIG))

# Phase 4: Darwinian
log('Phase 4: Darwinian Evolution')
for sym in list(all_results):
    all_results[sym].extend(darwinian(dfs[sym], folds, sym, all_results[sym], OVERFITTING_CONFIG, DARWINIAN_GENERATIONS, DARWINIAN_POPULATION))

elapsed = time.time() - t0
log(f'Done: {elapsed:.0f}s ({elapsed/60:.1f} min)')

---
## 6. Display Results

In [ ]:
from google.colab import data_table
data_table.enable_dataframe_formatter()

rows = []
for sym, results in all_results.items():
    validated = sorted([r for r in results if not r.rejected and not r.is_coarse_only],
                        key=lambda r: r.survivor_score, reverse=True)
    for i, cr in enumerate(validated[:20]):
        rows.append({
            'symbol': sym, 'rank': i+1,
            'survivor': round(cr.survivor_score, 2),
            'oos_sharpe': round(cr.oos_sharpe, 2),
            'consistency': f'{cr.oos_consistency:.0%}',
            'pf': round(cr.oos_pf, 2),
            'max_dd': f'{cr.oos_max_dd:.1f}%',
            'trades_fold': round(cr.oos_avg_trades_per_fold, 0),
            'fragility': round(cr.fragility, 2),
            'params': json.dumps(cr.params, separators=(', ', ':')),
        })

if rows:
    pd.DataFrame(rows).style.background_gradient(subset=['survivor'], cmap='Greens')
else:
    print('No validated candidates')

In [ ]:
# Per-symbol detail
for sym, results in all_results.items():
    prod = next((r for r in results if r.params == PRODUCTION_CONFIG), None)
    validated = [r for r in results if not r.rejected and not r.is_coarse_only]
    if not validated: print(f'\n{sym}: no validated candidates'); continue
    best = max(validated, key=lambda r: r.survivor_score)
    print(f'\n{"="*60}')
    print(f'{sym} — Best: survivor={best.survivor_score:.2f} sharpe={best.oos_sharpe:+.2f} cons={best.oos_consistency:.0%}')
    if prod:
        print(f'  Production: sharpe={prod.oos_sharpe:+.2f} cons={prod.oos_consistency:.0%} survivor={prod.survivor_score:.2f}')
        delta = best.survivor_score - prod.survivor_score
        print(f'  Delta: {delta:+.2f} ({"IMPROVE" if delta > 0 else "WORSE"})')
    print(f'  Params: {json.dumps(best.params)}')
    print(f'  Trades/fold: {best.oos_avg_trades_per_fold:.0f} | DD: {best.oos_max_dd:.1f}% | PF: {best.oos_pf:.1f}')
    print(f'  Exits: {dict(best.oos_exit_reasons)}')
    print(f'\n  {"Fold":>4} {"IS":>8} {"OOS":>8} {"PnL":>8} {"Trades":>7}')
    for fd in best.folds:
        raw = f' (raw:{fd["oos_sharpe_raw"]:+.0f})' if abs(fd.get('oos_sharpe_raw', 0)) > 100 else ''
        ok = '+' if fd['oos_sharpe'] > 0 else '-'
        print(f'  {fd["fold"]:4d} {fd["is_sharpe"]:+8.2f} {fd["oos_sharpe"]:+8.2f}{raw} {fd["oos_pnl"]:+7.2f}% {fd["oos_trades"]:5d} {ok}')

---
## 7. Save to Google Drive

In [ ]:
date_str = datetime.now(timezone.utc).strftime('%Y-%m-%d')
out = os.path.join(RESULTS_DIR, date_str)
os.makedirs(out, exist_ok=True)

# JSON
json_data = {'run_at': datetime.now(timezone.utc).isoformat(), 'runtime_s': elapsed,
    'folds': len(folds), 'symbols': list(all_results.keys()), 'top': []}
for sym, results in all_results.items():
    for cr in sorted([r for r in results if not r.rejected and not r.is_coarse_only],
                      key=lambda r: r.survivor_score, reverse=True)[:20]:
        json_data['top'].append({'symbol': cr.symbol, 'params': cr.params,
            'survivor': round(cr.survivor_score, 4), 'sharpe': round(cr.oos_sharpe, 4),
            'consistency': round(cr.oos_consistency, 4), 'max_dd': round(cr.oos_max_dd, 4),
            'fragility': round(cr.fragility, 4), 'trades': round(cr.oos_avg_trades_per_fold, 1),
            'exits': cr.oos_exit_reasons, 'folds': cr.folds})
with open(os.path.join(out, 'summary.json'), 'w') as f: json.dump(json_data, f, indent=2, default=str)

# Full results
full = {}
for sym, results in all_results.items():
    full[sym] = [asdict(r) for r in sorted(results, key=lambda r: r.survivor_score, reverse=True)[:50]]
with open(os.path.join(out, 'full_results.json'), 'w') as f: json.dump(full, f, indent=2, default=str)

# Markdown report
lines = [f'# Night Shift Report — {date_str}',
    f'\n**Runtime:** {elapsed:.0f}s ({elapsed/60:.1f} min) | **Folds:** {len(folds)}',
    f'**Aggregation:** Median OOS Sharpe, winsorized at +/-{SHARPE_CAP}']
for sym, results in all_results.items():
    prod = next((r for r in results if r.params == PRODUCTION_CONFIG), None)
    valid = sorted([r for r in results if not r.rejected and not r.is_coarse_only],
                   key=lambda r: r.survivor_score, reverse=True)
    if not valid: continue
    b = valid[0]
    lines += [f'\n## {sym}', f'**Best:** Sharpe={b.oos_sharpe:+.2f} cons={b.oos_consistency:.0%} survivor={b.survivor_score:.2f}']
    if prod: lines.append(f'**Production:** Sharpe={prod.oos_sharpe:+.2f} cons={prod.oos_consistency:.0%}')
    lines += [f'\n```json\n{json.dumps(b.params, indent=2)}\n```',
        f'Trades/fold: {b.oos_avg_trades_per_fold:.0f} | DD: {b.oos_max_dd:.1f}% | PF: {b.oos_pf:.1f}',
        f'\n| Fold | IS | OOS | PnL | Trades |', '|------|------|------|------|--------|']
    for fd in b.folds:
        lines.append(f'| {fd["fold"]} | {fd["is_sharpe"]:+.2f} | {fd["oos_sharpe"]:+.2f} | {fd["oos_pnl"]:+.2f}% | {fd["oos_trades"]} |')
with open(os.path.join(out, 'report.md'), 'w') as f: f.write('\n'.join(lines))

print(f'Saved to Google Drive: {out}/')
print(f'  report.md  —  human-readable summary')
print(f'  summary.json  —  top 20 candidates')
print(f'  full_results.json  —  top 50 per symbol')
print(f'\nFiles will sync to your local machine via Google Drive.')